In [16]:
import pandas as pd
import datetime as dt

In [17]:
price_df = pd.read_csv("D:\egg_price_procject\output\egg_price_data.csv")
festival_df = pd.read_csv("D:\egg_price_procject\output\india_festival_dates.csv")
weather_df = pd.read_csv("D:\egg_price_procject\output\india_weather_data.csv")

In [18]:
price_df.head()

,Category,City,Price,Day,Month,Year,Date
0,NECC SUGGESTED EGG PRICES,Ahmedabad,545.0,12,5,2026,2026-05-12
1,NECC SUGGESTED EGG PRICES,Ajmer,510.0,12,5,2026,2026-05-12
2,NECC SUGGESTED EGG PRICES,Barwala,518.0,12,5,2026,2026-05-12
3,NECC SUGGESTED EGG PRICES,Bengaluru (CC),590.0,12,5,2026,2026-05-12
4,NECC SUGGESTED EGG PRICES,Brahmapur (OD),550.0,12,5,2026,2026-05-12


In [19]:
price_df.shape

(16897, 7)

In [20]:
# Festival data
festival_df.head()

,Date,festival_name,Year,Month,Day
0,2016-05-21,Buddha Purnima,2016,5,21
1,2016-07-07,Id-ul-Fitr,2016,7,7
2,2016-08-15,Independence Day,2016,8,15
3,2016-08-25,Janmashtami,2016,8,25
4,2016-09-13,Bakrid,2016,9,13


In [21]:
festival_df.shape

(160, 5)

In [22]:
# weather data
weather_df.head()

,state,date,tavg,tmin,tmax,prcp,latitude,longitude
0,Andhra Pradesh,2016-05-12,32.7,28.5,39.4,0.9,16.5062,80.648
1,Andhra Pradesh,2016-05-13,33.6,28.2,40.7,0.2,16.5062,80.648
2,Andhra Pradesh,2016-05-14,33.5,29.1,39.7,0.1,16.5062,80.648
3,Andhra Pradesh,2016-05-15,32.0,28.8,36.9,0.0,16.5062,80.648
4,Andhra Pradesh,2016-05-16,32.3,28.2,38.7,0.5,16.5062,80.648


In [23]:
weather_df.shape

(87672, 8)

### feature update in price_df

In [24]:
# converting date column to date time formate
price_df['Date'] = pd.to_datetime(price_df.Date)
# removing all extra text from city name
price_df['City'] = price_df['City'] = price_df['City'].str.split().str[0]

In [25]:
price_df.City.unique()

array(['Ahmedabad', 'Ajmer', 'Barwala', 'Bengaluru', 'Brahmapur',
       'Chennai', 'Chittoor', 'Delhi', 'E.Godavari', 'Hospet',
       'Hyderabad', 'Jabalpur', 'Kolkata', 'Ludhiana', 'Mumbai', 'Mysuru',
       'Namakkal', 'Pune', 'Raipur', 'Surat', 'Vijayawada', 'Vizag',
       'W.Godavari', 'Warangal', 'Allahabad', 'Bhopal', 'Indore',
       'Kanpur', 'Luknow', 'Muzaffurpur', 'Nagpur', 'Patna', 'Ranchi',
       'Varanasi'], dtype=object)

In [26]:
price_df.head()

,Category,City,Price,Day,Month,Year,Date
0,NECC SUGGESTED EGG PRICES,Ahmedabad,545.0,12,5,2026,2026-05-12
1,NECC SUGGESTED EGG PRICES,Ajmer,510.0,12,5,2026,2026-05-12
2,NECC SUGGESTED EGG PRICES,Barwala,518.0,12,5,2026,2026-05-12
3,NECC SUGGESTED EGG PRICES,Bengaluru,590.0,12,5,2026,2026-05-12
4,NECC SUGGESTED EGG PRICES,Brahmapur,550.0,12,5,2026,2026-05-12


In [27]:
# getting data for only Barwala
df_Barwala = price_df[price_df['City']=='Barwala']
df_Barwala = df_Barwala.sort_values(['Date'])

In [28]:
df_Barwala.Date.max()

Timestamp('2026-05-12 00:00:00')

In [29]:
df_Barwala.dtypes

Category            object
City                object
Price              float64
Day                  int64
Month                int64
Year                 int64
Date        datetime64[ns]
dtype: object

In [30]:
#creating price percentange change on yersterday
df_Barwala['daily_change_pct'] = (
    df_Barwala.groupby('City')['Price']
    .pct_change() * 100
)

In [31]:
# function which make rating on the basis of percentage change of price
def daily_market_rating(change):

    if pd.isna(change):
        return 0

    if change >= 10:
        return 5

    elif change >= 7:
        return 4

    elif change >= 5:
        return 3

    elif change >= 3:
        return 2

    elif change >= 1:
        return 1

    elif change <= -10:
        return -5

    elif change <= -7:
        return -4

    elif change <= -5:
        return -3

    elif change <= -3:
        return -2

    elif change <= -1:
        return -1

    else:
        return 0
    




# creating market rating column on the baisis of percentage change
df_Barwala['market_rating'] = (
    df_Barwala['daily_change_pct']
    .apply(daily_market_rating)
)

In [32]:
# creating a column of lag of 1 day price
df_Barwala['lag_1'] = (
    df_Barwala.groupby('City')['Price']
    .shift(1)
)

In [33]:
# creating a column of lag of 7 day price
df_Barwala['lag_7'] = (
    df_Barwala.groupby('City')['Price']
    .shift(7)
)

In [34]:
df_Barwala.head(10)

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7
16865,NECC SUGGESTED EGG PRICES,Barwala,574.0,1,1,2025,2025-01-01,NaN,0,NaN,NaN
16831,NECC SUGGESTED EGG PRICES,Barwala,560.0,2,1,2025,2025-01-02,-2.439024,-1,574.0,NaN
16797,NECC SUGGESTED EGG PRICES,Barwala,560.0,3,1,2025,2025-01-03,0.000000,0,560.0,NaN
16763,NECC SUGGESTED EGG PRICES,Barwala,560.0,4,1,2025,2025-01-04,0.000000,0,560.0,NaN
16729,NECC SUGGESTED EGG PRICES,Barwala,505.0,5,1,2025,2025-01-05,-9.821429,-4,560.0,NaN
16695,NECC SUGGESTED EGG PRICES,Barwala,505.0,6,1,2025,2025-01-06,0.000000,0,505.0,NaN
16661,NECC SUGGESTED EGG PRICES,Barwala,505.0,7,1,2025,2025-01-07,0.000000,0,505.0,NaN
16627,NECC SUGGESTED EGG PRICES,Barwala,515.0,8,1,2025,2025-01-08,1.980198,1,505.0,574.0
16593,NECC SUGGESTED EGG PRICES,Barwala,520.0,9,1,2025,2025-01-09,0.970874,0,515.0,560.0
16559,NECC SUGGESTED EGG PRICES,Barwala,520.0,10,1,2025,2025-01-10,0.000000,0,520.0,560.0


### feature updates in festival_df

In [35]:
festival_df.head()

,Date,festival_name,Year,Month,Day
0,2016-05-21,Buddha Purnima,2016,5,21
1,2016-07-07,Id-ul-Fitr,2016,7,7
2,2016-08-15,Independence Day,2016,8,15
3,2016-08-25,Janmashtami,2016,8,25
4,2016-09-13,Bakrid,2016,9,13


In [36]:
festival_df= festival_df[['Date','festival_name']]
festival_df['Date'] = pd.to_datetime(festival_df.Date)

In [37]:
festival_df.Date.max()

Timestamp('2026-05-01 00:00:00')

In [38]:
# Merging of price_df and festival_df on date 
data_barwala = df_Barwala.merge(
    festival_df,
    on='Date',
    how='left'
)

In [39]:
data_barwala.head()

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7,festival_name
0,NECC SUGGESTED EGG PRICES,Barwala,574.0,1,1,2025,2025-01-01,NaN,0,NaN,NaN,NaN
1,NECC SUGGESTED EGG PRICES,Barwala,560.0,2,1,2025,2025-01-02,-2.439024,-1,574.0,NaN,NaN
2,NECC SUGGESTED EGG PRICES,Barwala,560.0,3,1,2025,2025-01-03,0.000000,0,560.0,NaN,NaN
3,NECC SUGGESTED EGG PRICES,Barwala,560.0,4,1,2025,2025-01-04,0.000000,0,560.0,NaN,NaN
4,NECC SUGGESTED EGG PRICES,Barwala,505.0,5,1,2025,2025-01-05,-9.821429,-4,560.0,NaN,NaN


In [40]:
data_barwala.Date.max()

Timestamp('2026-05-12 00:00:00')

In [41]:
data_barwala.festival_name.value_counts()

Republic Day                2
Maha Shivaratri             2
Mahavir Jayanti             2
Good Friday                 2
Buddha Purnima              2
Id-ul-Fitr                  1
Bakrid                      1
Muharram                    1
Independence Day            1
Janmashtami                 1
Milad-un-Nabi               1
Dussehra; Gandhi Jayanti    1
Diwali                      1
Guru Nanak Jayanti          1
Christmas                   1
Id-ul-Fitr (estimated)      1
Name: festival_name, dtype: int64

### feature updates in weather_df

In [42]:
weather_df.head()

,state,date,tavg,tmin,tmax,prcp,latitude,longitude
0,Andhra Pradesh,2016-05-12,32.7,28.5,39.4,0.9,16.5062,80.648
1,Andhra Pradesh,2016-05-13,33.6,28.2,40.7,0.2,16.5062,80.648
2,Andhra Pradesh,2016-05-14,33.5,29.1,39.7,0.1,16.5062,80.648
3,Andhra Pradesh,2016-05-15,32.0,28.8,36.9,0.0,16.5062,80.648
4,Andhra Pradesh,2016-05-16,32.3,28.2,38.7,0.5,16.5062,80.648


In [43]:
weather_df['state'].unique()

array(['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar',
       'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh',
       'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh',
       'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland',
       'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu',
       'Telangana'], dtype=object)

In [44]:
#form name in state
to_state = {
    'Ahmedabad': 'Gujarat',
    'Ajmer': 'Rajasthan',
    'Barwala': 'Haryana',
    'Bengaluru': 'Karnataka',
    'Brahmapur': 'Odisha',
    'Chennai': 'Tamil Nadu',
    'Delhi': 'Delhi',
    'E.Godavari': 'Andhra Pradesh',
    'Hospet': 'Karnataka',
    'Jabalpur': 'Madhya Pradesh',
    'Kolkata': 'West Bengal',
    'Ludhiana': 'Punjab',
    'Mumbai': 'Maharashtra',
    'Mysuru': 'Karnataka',
    'Namakkal': 'Tamil Nadu',
    'Pune': 'Maharashtra',
    'Raipur': 'Chhattisgarh',
    'Surat': 'Gujarat',
    'Vijayawada': 'Andhra Pradesh',
    'Vizag': 'Andhra Pradesh',
    'W.Godavari': 'Andhra Pradesh',
    'Warangal': 'Telangana',
    'Allahabad': 'Uttar Pradesh',
    'Bhopal': 'Madhya Pradesh',
    'Indore': 'Madhya Pradesh',
    'Kanpur': 'Uttar Pradesh',
    'Luknow': 'Uttar Pradesh',
    'Muzaffurpur': 'Bihar',
    'Nagpur': 'Maharashtra',
    'Patna': 'Bihar',
    'Ranchi': 'Jharkhand',
    'Varanasi': 'Uttar Pradesh',
    'Chittoor': 'Andhra Pradesh',
    'Hyderabad': 'Telangana'
}

In [45]:
weather_df_Barwala = weather_df[['date','tmax','prcp']][weather_df['state']==to_state['Barwala']]
weather_df_Barwala['date'] = pd.to_datetime(weather_df_Barwala.date)

In [46]:
weather_df_Barwala.head()

,date,tmax,prcp
25571,2016-05-12,37.3,0.0
25572,2016-05-13,39.5,0.5
25573,2016-05-14,40.9,0.0
25574,2016-05-15,41.1,0.0
25575,2016-05-16,41.3,0.2


In [52]:
weather_df_Barwala.date.min()

Timestamp('2016-05-12 00:00:00')

In [47]:
weather_df_Barwala.date.max()

Timestamp('2026-05-12 00:00:00')

In [48]:
weather_df_Barwala.dtypes

date    datetime64[ns]
tmax           float64
prcp           float64
dtype: object

In [49]:
# Merging of data_barwala and weather_df_Barwala on date 
data_barwala = data_barwala.merge(
    weather_df_Barwala,
    left_on='Date',
    right_on='date',
    how='left'
)

In [50]:
data_barwala.head(10)

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7,festival_name,date,tmax,prcp
0,NECC SUGGESTED EGG PRICES,Barwala,574.0,1,1,2025,2025-01-01,NaN,0,NaN,NaN,NaN,2025-01-01,16.8,0.0
1,NECC SUGGESTED EGG PRICES,Barwala,560.0,2,1,2025,2025-01-02,-2.439024,-1,574.0,NaN,NaN,2025-01-02,18.4,0.0
2,NECC SUGGESTED EGG PRICES,Barwala,560.0,3,1,2025,2025-01-03,0.000000,0,560.0,NaN,NaN,2025-01-03,20.5,0.0
3,NECC SUGGESTED EGG PRICES,Barwala,560.0,4,1,2025,2025-01-04,0.000000,0,560.0,NaN,NaN,2025-01-04,21.7,0.0
4,NECC SUGGESTED EGG PRICES,Barwala,505.0,5,1,2025,2025-01-05,-9.821429,-4,560.0,NaN,NaN,2025-01-05,19.1,0.0
5,NECC SUGGESTED EGG PRICES,Barwala,505.0,6,1,2025,2025-01-06,0.000000,0,505.0,NaN,NaN,2025-01-06,18.5,0.7
6,NECC SUGGESTED EGG PRICES,Barwala,505.0,7,1,2025,2025-01-07,0.000000,0,505.0,NaN,NaN,2025-01-07,18.0,0.0
7,NECC SUGGESTED EGG PRICES,Barwala,515.0,8,1,2025,2025-01-08,1.980198,1,505.0,574.0,NaN,2025-01-08,18.0,0.0
8,NECC SUGGESTED EGG PRICES,Barwala,520.0,9,1,2025,2025-01-09,0.970874,0,515.0,560.0,NaN,2025-01-09,17.3,0.0
9,NECC SUGGESTED EGG PRICES,Barwala,520.0,10,1,2025,2025-01-10,0.000000,0,520.0,560.0,NaN,2025-01-10,16.1,0.0


### Final feature engeeniring

In [51]:
#dropoing some duplicate columns
data_barwala = data_barwala.drop(columns=[
    'date_y',
    'tmax_y',
    'prcp_y'
])

KeyError: "['date_y', 'tmax_y', 'prcp_y'] not found in axis"

In [ ]:
# renaimig
data_barwala = data_barwala.rename(columns={
    'date_x': 'weather_date',
    'tmax_x': 'tmax',
    'prcp_x': 'prcp'
})

In [ ]:
# droping nan value from lag features
data_barwala = data_barwala.dropna(
    subset=['lag_1', 'lag_7']
)

In [ ]:
#converting festival into 1 and 0
data_barwala['is_festival'] = (
    data_barwala['festival_name']
    .notna()
).astype(int)

In [ ]:
# week day number
data_barwala['day_of_week'] = (
    data_barwala['Date'].dt.dayofweek
)

In [ ]:
data_barwala.head()

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7,festival_name,weather_date,tmax,prcp,is_festival,day_of_week
7,NECC SUGGESTED EGG PRICES,Barwala,480.0,12,5,2021,2021-05-12,7.865169,4,445.0,352.0,NaN,2021-05-12,32.3,1.2,0,2
8,NECC SUGGESTED EGG PRICES,Barwala,495.0,13,5,2021,2021-05-13,3.125000,2,480.0,380.0,NaN,2021-05-13,28.7,3.6,0,3
9,NECC SUGGESTED EGG PRICES,Barwala,495.0,14,5,2021,2021-05-14,0.000000,0,495.0,406.0,Id-ul-Fitr,2021-05-14,32.2,0.0,1,4
10,NECC SUGGESTED EGG PRICES,Barwala,500.0,15,5,2021,2021-05-15,1.010101,1,495.0,430.0,NaN,2021-05-15,34.4,0.0,0,5
11,NECC SUGGESTED EGG PRICES,Barwala,503.0,16,5,2021,2021-05-16,0.600000,0,500.0,442.0,NaN,2021-05-16,35.8,0.0,0,6


In [ ]:
# feature i am going to take for models

features = [
    'lag_1',
    'lag_7',
    'market_rating',
    'tmax',
    'prcp',
    'is_festival',
    'Month',
    'day_of_week'
]

In [ ]:
festival_df = pd.read_csv("D:\egg_price_procject\output\india_festival_dates.csv")

In [ ]:
festival_df['Date'] = pd.to_datetime(festival_df['Date'])

In [ ]:
festival_df.dtypes

Date             datetime64[ns]
festival_name            object
Year                      int64
Month                     int64
Day                       int64
dtype: object

In [ ]:
merged_df = df.merge(
    festival_df,
    on='Date',
    how='left'
)

In [ ]:
merged_df.head()

,Category,City,Price,Day_x,Month_x,Year_x,Date,daily_change_pct,market_rating,lag_1,lag_7,festival_name,Year_y,Month_y,Day_y
0,NECC SUGGESTED EGG PRICES,Barwala,352.0,5,5,2021,2021-05-05,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN
1,NECC SUGGESTED EGG PRICES,Barwala,380.0,6,5,2021,2021-05-06,7.954545,4,352.0,NaN,NaN,NaN,NaN,NaN
2,NECC SUGGESTED EGG PRICES,Barwala,406.0,7,5,2021,2021-05-07,6.842105,3,380.0,NaN,NaN,NaN,NaN,NaN
3,NECC SUGGESTED EGG PRICES,Barwala,430.0,8,5,2021,2021-05-08,5.911330,3,406.0,NaN,NaN,NaN,NaN,NaN
4,NECC SUGGESTED EGG PRICES,Barwala,442.0,9,5,2021,2021-05-09,2.790698,1,430.0,NaN,NaN,NaN,NaN,NaN


In [ ]:
festival.head()

,Date,festival_name,Year,Month,Day
0,2021-05-14,Id-ul-Fitr,2021,5,14
1,2021-05-26,Buddha Purnima,2021,5,26
2,2021-07-21,Bakrid,2021,7,21
3,2021-08-15,Independence Day,2021,8,15
4,2021-08-20,Muharram,2021,8,20


In [ ]:
df_2026 = df[data['Year'] == 2022]

fig = px.line(
    df_2026,
    x='Date',
    y='Price',
    title='Egg Price Trend - 2026',
)

fig.show()

In [ ]:
df.head()

,Category,City,Price,Day,Month,Year,Date,daily_change_pct,market_rating,lag_1,lag_7
62081,NECC SUGGESTED EGG PRICES,Barwala,352.0,5,5,2021,2021-05-05,NaN,0,NaN,NaN
62047,NECC SUGGESTED EGG PRICES,Barwala,380.0,6,5,2021,2021-05-06,7.954545,4,352.0,NaN
62013,NECC SUGGESTED EGG PRICES,Barwala,406.0,7,5,2021,2021-05-07,6.842105,3,380.0,NaN
61979,NECC SUGGESTED EGG PRICES,Barwala,430.0,8,5,2021,2021-05-08,5.911330,3,406.0,NaN
61945,NECC SUGGESTED EGG PRICES,Barwala,442.0,9,5,2021,2021-05-09,2.790698,1,430.0,NaN


In [ ]:
data.City.unique()

array(['Ahmedabad', 'Ajmer', 'Barwala', 'Bengaluru', 'Brahmapur',
       'Chennai', 'Delhi', 'E.Godavari', 'Hospet', 'Jabalpur', 'Kolkata',
       'Ludhiana', 'Mumbai', 'Mysuru', 'Namakkal', 'Pune', 'Raipur',
       'Surat', 'Vijayawada', 'Vizag', 'W.Godavari', 'Warangal',
       'Allahabad', 'Bhopal', 'Indore', 'Kanpur', 'Luknow', 'Muzaffurpur',
       'Nagpur', 'Patna', 'Ranchi', 'Varanasi', 'Chittoor', 'Hyderabad'],
      dtype=object)

In [ ]:
weather = pd.read_csv("D:\egg_price_procject\output\india_weather_data.csv")

In [ ]:
weather.head()

,state,date,tavg,tmin,tmax,prcp,latitude,longitude
0,Andaman And Nicobar Islands,2021-05-06,28.4,26.0,31.0,9.2,11.6234,92.7265
1,Andaman And Nicobar Islands,2021-05-07,27.5,26.2,28.5,19.9,11.6234,92.7265
2,Andaman And Nicobar Islands,2021-05-08,26.2,25.6,27.0,16.2,11.6234,92.7265
3,Andaman And Nicobar Islands,2021-05-09,27.7,25.3,30.3,3.6,11.6234,92.7265
4,Andaman And Nicobar Islands,2021-05-10,27.9,25.1,30.5,3.8,11.6234,92.7265


In [ ]:
weather.date.max()

'2026-05-06'

In [ ]:
weather.state.unique()

array(['Andaman And Nicobar Islands', 'Andhra Pradesh',
       'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
       'Chhattisgarh', 'Dadra And Nagar Haveli And Daman And Diu',
       'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh',
       'Jammu And Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Ladakh',
       'Lakshadweep', 'Madhya Pradesh', 'Maharashtra', 'Manipur',
       'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Puducherry',
       'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana',
       'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal'],
      dtype=object)

In [ ]:
delhi_weather = weather[weather['state']=='Delhi']

In [ ]:
delhi_weather.head()

,state,date,tavg,tmin,tmax,prcp,latitude,longitude
14616,Delhi,2021-05-06,29.3,24.0,38.7,6.1,28.6139,77.209
14617,Delhi,2021-05-07,28.3,21.7,36.2,4.1,28.6139,77.209
14618,Delhi,2021-05-08,31.1,23.6,37.4,0.0,28.6139,77.209
14619,Delhi,2021-05-09,31.0,25.2,37.9,0.0,28.6139,77.209
14620,Delhi,2021-05-10,30.5,25.0,36.2,0.4,28.6139,77.209


In [ ]:
delhi_weather[delhi_weather['date']==delhi_weather.date.max()]

,state,date,tavg,tmin,tmax,prcp,latitude,longitude
16442,Delhi,2026-05-06,28.8,21.9,35.2,0.0,28.6139,77.209


In [ ]:
import pandas as pd

In [ ]:
cities = {
    "Ahmedabad": (23.0225, 72.5714),
    "Ajmer": (26.4499, 74.6399),
    "Barwala": (29.3670, 75.9080),
    "Bengaluru": (12.9716, 77.5946),
    "Brahmapur": (19.3149, 84.7941),
    "Chennai": (13.0827, 80.2707),
    "Delhi": (28.6139, 77.2090),
    "E.Godavari": (16.9891, 82.2475),
    "Hospet": (15.2689, 76.3909),
    "Jabalpur": (23.1815, 79.9864),
    "Kolkata": (22.5726, 88.3639),
    "Ludhiana": (30.9000, 75.8573),
    "Mumbai": (19.0760, 72.8777),
    "Mysuru": (12.2958, 76.6394),
    "Namakkal": (11.2189, 78.1674),
    "Pune": (18.5204, 73.8567),
    "Raipur": (21.2514, 81.6296),
    "Surat": (21.1702, 72.8311),
    "Vijayawada": (16.5062, 80.6480),
    "Vizag": (17.6868, 83.2185),
    "W.Godavari": (16.7107, 81.0952),
    "Warangal": (17.9689, 79.5941),
    "Allahabad": (25.4358, 81.8463),
    "Bhopal": (23.2599, 77.4126),
    "Indore": (22.7196, 75.8577),
    "Kanpur": (26.4499, 80.3319),
    "Luknow": (26.8467, 80.9462),
    "Muzaffurpur": (26.1209, 85.3647),
    "Nagpur": (21.1458, 79.0882),
    "Patna": (25.5941, 85.1376),
    "Ranchi": (23.3441, 85.3096),
    "Varanasi": (25.3176, 82.9739),
    "Chittoor": (13.2172, 79.1003),
    "Hyderabad": (17.3850, 78.4867)
}

In [ ]:

from datetime import datetime

In [ ]:
datetime.today().date()

datetime.date(2026, 5, 6)

In [ ]:
from meteostat import Point, Daily

In [ ]:
from meteostat import Point, Daily
print("Working ✅")
import pandas as pd

start = datetime(2015, 1, 1) # 10 years
end = datetime(2026, 5, 6)

all_data = []

for city, (lat, lon) in cities.items():
    print(f"Fetching data for {city}...")

    location = Point(lat, lon)
    data = Daily(location, start, end).fetch()

    data['city'] = city
    data.reset_index(inplace=True)

    all_data.append(data)

# Combine all cities
final_df = pd.concat(all_data)

# Save file
final_df.to_csv("india_20yr_temperature.csv", index=False)

print("✅ Data saved successfully!")

Working ✅
Fetching data for Ahmedabad...
Fetching data for Ajmer...
Fetching data for Barwala...
Fetching data for Bengaluru...
Fetching data for Brahmapur...
Fetching data for Chennai...
Fetching data for Delhi...
Fetching data for E.Godavari...
Fetching data for Hospet...
Fetching data for Jabalpur...
Fetching data for Kolkata...
Fetching data for Ludhiana...
Fetching data for Mumbai...
Fetching data for Mysuru...
Fetching data for Namakkal...
Fetching data for Pune...
Fetching data for Raipur...
Fetching data for Surat...
Fetching data for Vijayawada...
Fetching data for Vizag...
Fetching data for W.Godavari...
Fetching data for Warangal...
Fetching data for Allahabad...
Fetching data for Bhopal...
Fetching data for Indore...
Fetching data for Kanpur...
Fetching data for Luknow...
Fetching data for Muzaffurpur...
Fetching data for Nagpur...
Fetching data for Patna...
Fetching data for Ranchi...
Fetching data for Varanasi...
Fetching data for Chittoor...
Fetching data for Hyderabad..

In [ ]:
t_data = pd.read_csv("india_20yr_temperature.csv")

In [ ]:
t_data.head()

,time,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun,city,index
0,2015-01-01,19.6,NaN,24.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ahmedabad,NaN
1,2015-01-02,19.6,NaN,25.0,0.8,NaN,NaN,NaN,NaN,NaN,NaN,Ahmedabad,NaN
2,2015-01-03,18.8,NaN,25.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,Ahmedabad,NaN
3,2015-01-04,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ahmedabad,NaN
4,2015-01-05,17.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Ahmedabad,NaN


In [ ]:
t_data['time'] = pd.to_datetime(t_data['time'])

In [ ]:
t_data.dtypes

time     datetime64[ns]
tavg            float64
tmin            float64
tmax            float64
prcp            float64
snow            float64
wdir            float64
wspd            float64
wpgt            float64
pres            float64
tsun            float64
city             object
index           float64
dtype: object

In [ ]:
t_data['time'].max()

Timestamp('2026-03-19 00:00:00')

In [ ]:
t_data.sort_values(['time'],ascending=False)

,time,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun,city,index
118826,2026-03-19,26.8,21.9,32.4,NaN,NaN,120.0,2.4,NaN,1009.7,NaN,Hyderabad,NaN
93122,2026-03-19,24.7,19.0,31.2,NaN,NaN,324.0,4.1,NaN,1011.1,NaN,Luknow,NaN
49766,2026-03-19,27.7,23.8,32.1,NaN,NaN,NaN,4.3,NaN,1010.2,NaN,Mumbai,NaN
42091,2026-03-19,26.5,22.8,30.9,NaN,NaN,171.0,2.7,NaN,1010.2,NaN,Kolkata,NaN
99113,2026-03-19,26.4,20.3,32.9,NaN,NaN,115.0,6.0,NaN,1010.3,NaN,Nagpur,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
66430,2015-01-01,26.6,20.3,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Vijayawada,NaN
19771,2015-01-01,26.1,21.9,30.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Chennai,NaN
107290,2015-01-01,15.2,NaN,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Varanasi,NaN
99114,2015-01-01,19.8,NaN,25.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Patna,NaN


In [ ]:
import requests
import pandas as pd
import time

all_data = []
failed_cities = []

start_date = "2015-01-01"
end_date = "2025-01-01"

for city, (lat, lon) in cities.items():

    success = False

    while not success:

        print(f"Fetching {city}...")

        url = (
            f"https://archive-api.open-meteo.com/v1/archive?"
            f"latitude={lat}"
            f"&longitude={lon}"
            f"&start_date={start_date}"
            f"&end_date={end_date}"
            f"&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum"
            f"&timezone=auto"
        )

        try:

            response = requests.get(url, timeout=30)
            data = response.json()

            # Rate limit hit
            if 'daily' not in data:

                print(f"⚠️ API limit hit for {city}")
                print("⏳ Waiting 65 seconds...\n")

                time.sleep(65)
                continue

            daily = data['daily']

            df = pd.DataFrame({
                'date': daily['time'],
                'tavg': daily['temperature_2m_mean'],
                'tmin': daily['temperature_2m_min'],
                'tmax': daily['temperature_2m_max'],
                'prcp': daily['precipitation_sum']
            })

            df['city'] = city

            all_data.append(df)

            print(f"✅ Done {city}\n")

            success = True

            time.sleep(2)

        except Exception as e:

            print(f"❌ Error for {city}: {e}")

            failed_cities.append(city)

            break

# Combine all cities
final_df = pd.concat(all_data)

# Save CSV
final_df.to_csv("india_weather_10yr.csv", index=False)

print("✅ Weather dataset saved successfully!")

print("\nFailed Cities:")
print(failed_cities)

Fetching Ahmedabad...
✅ Done Ahmedabad

Fetching Ajmer...
✅ Done Ajmer

Fetching Barwala...
✅ Done Barwala

Fetching Bengaluru...
✅ Done Bengaluru

Fetching Brahmapur...
✅ Done Brahmapur

Fetching Chennai...
✅ Done Chennai

Fetching Delhi...
⚠️ API limit hit for Delhi
⏳ Waiting 65 seconds...

Fetching Delhi...
✅ Done Delhi

Fetching E.Godavari...
✅ Done E.Godavari

Fetching Hospet...
✅ Done Hospet

Fetching Jabalpur...
✅ Done Jabalpur

Fetching Kolkata...
✅ Done Kolkata

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seconds...

Fetching Ludhiana...
⚠️ API limit hit for Ludhiana
⏳ Waiting 65 seco

In [ ]:
indian_t = pd.read_csv("india_weather_10yr.csv")

In [ ]:
indian_t.head()

,date,tavg,tmin,tmax,prcp,city
0,2015-01-01,20.3,16.9,24.3,0.0,Ahmedabad
1,2015-01-02,20.4,17.3,23.6,0.0,Ahmedabad
2,2015-01-03,19.7,15.1,25.4,0.0,Ahmedabad
3,2015-01-04,19.8,14.5,26.3,0.0,Ahmedabad
4,2015-01-05,19.0,13.7,25.4,0.0,Ahmedabad


In [ ]:
indian_t.groupby('city')['tavg'].mean()

city
Ahmedabad      27.317050
Ajmer          24.906459
Allahabad      25.809086
Barwala        24.384072
Bengaluru      23.308785
Bhopal         25.152025
Brahmapur      26.754461
Chennai        28.007827
Chittoor       26.930843
Delhi          24.529611
E.Godavari     27.695457
Hospet         26.472852
Hyderabad      25.841981
Indore         25.024548
Jabalpur       25.256951
Kanpur         25.350301
Kolkata        26.051642
Ludhiana       23.036864
Luknow         25.044964
Mumbai         26.707033
Muzaffurpur    25.068582
Mysuru         24.094527
Nagpur         26.793760
Namakkal       28.265079
Patna          25.482184
Pune           24.723344
Raipur         26.268090
Ranchi         23.141133
Surat          27.197400
Varanasi       25.757499
Vijayawada     28.375151
Vizag          27.122469
W.Godavari     28.079365
Warangal       27.065599
Name: tavg, dtype: float64

In [ ]:
indian_t['date'].max()

'2025-01-01'

In [ ]:
indian_t.city.unique()

array(['Ahmedabad', 'Ajmer', 'Barwala', 'Bengaluru', 'Brahmapur',
       'Chennai', 'Delhi', 'E.Godavari', 'Hospet', 'Jabalpur', 'Kolkata',
       'Ludhiana', 'Mumbai', 'Mysuru', 'Namakkal', 'Pune', 'Raipur',
       'Surat', 'Vijayawada', 'Vizag', 'W.Godavari', 'Warangal',
       'Allahabad', 'Bhopal', 'Indore', 'Kanpur', 'Luknow', 'Muzaffurpur',
       'Nagpur', 'Patna', 'Ranchi', 'Varanasi', 'Chittoor', 'Hyderabad'],
      dtype=object)

In [ ]:
indian_t.isnull().sum()

date    0
tavg    0
tmin    0
tmax    0
prcp    0
city    0
dtype: int64

In [ ]:
indian_t[['tavg','date']][indian_t['city']=='Delhi'].max()

tavg          39.2
date    2025-01-01
dtype: object